Run this in the terminal to initialize GEE
conda create -n gee python=3.11 # run only if the environment has not been created
conda activate gee
conda install -c conda-forge mamba
mamba install -c conda-forge geemap pygis

In [10]:
%pip install -U "geemap[workshop]"

In [1]:
import ee
import geemap

In [2]:
ee.Authenticate()

True

In [3]:
ee.Initialize()

In [4]:
# create interactive maps
# 1. The first step here is to simply use a general open source map
m = geemap.Map()
m

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

In [7]:
# Since we are interested in Liberia, we use the lat and long of Liberia (For GEEmap, it is lat and long)
m = geemap.Map(center=[5.5, -8], zoom = 10, height = 600)
m

Map(center=[5.5, -8], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', tra…

To hide a control like control name, tool bar or draw functionality, set them to false

In [8]:
m = geemap.Map(data_ctrl=False, toolbar_ctrl=False, draw_ctrl = False)
m

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

## Working with basemaps. 

In [9]:
# By default, open street map is used as the basemap. We can change it to other basemaps available in geemap.
m = geemap.Map()
m

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

In [10]:
# Alternatively, we can create an empty map and then add basemaps later
m = geemap.Map(center=[5.5, -8], zoom = 7, height = 600)
# Better to use Geemap with ipyleaflet ipywidget instead of folium. This is because folium does not support dynamic map updates. Add the basemap to the empty map.
m.add_basemap("Esri.WorldTopoMap", show=False)
m.add_basemap("OpenTopoMap", show=False)
m.add_basemap("Esri.WorldTopoMap", show=False)
m.add_basemap("NLCD 2021 CONUS Land Cover",show=False)
m.add_basemap("FWS NWI Wetlands",show=False)
m

Map(center=[5.5, -8], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', tra…

In [11]:
# Show all the basemaps
basemaps = list(geemap.basemaps.keys())
basemaps
len(geemap.basemaps)


150

In [12]:
# Show the first 10 basemaps from the list of basemaps
basemaps[:10]

['OpenStreetMap',
 'Esri.WorldStreetMap',
 'Esri.WorldImagery',
 'Esri.WorldTopoMap',
 'FWS NWI Wetlands',
 'FWS NWI Wetlands Raster',
 'NLCD 2021 CONUS Land Cover',
 'NLCD 2019 CONUS Land Cover',
 'NLCD 2016 CONUS Land Cover',
 'NLCD 2013 CONUS Land Cover']

The second part is about Earth Engine. All data are stored on the server side and not on the client side.

In [13]:
image = ee.Image('USGS/SRTMGL1_003')
image

**Loading image from Google Earth**
Example here is to load the image collection of the Sentinel 2 surface reflectance

Collecting images. Here, we will use images from sentinnel 2 surface reflectance.

In [14]:
collection = ee.ImageCollection("COPERNICUS/S2_SR")

**To visualize the image collections**, we need to convert the *ImageCollection* to an **Image** by compositing all the images in the collection to a single image representing, for exaple, the min, max, median ror SD of the images. 

In [14]:
m = geemap.Map(center=[21.79, 70.87], zoom=3)
image = ee.Image("USGS/SRTMGL1_003")
vis_params = {
    "min": 0,
    "max": 6000,
    "palette": ["006633", "E5FFCC", "662A00", "D8D8D8", "F5F5F5"],  # 'terrain'
}
m.add_layer(image, vis_params, "SRTM")
m

Map(center=[21.79, 70.87], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

In [17]:
m = geemap.Map(center=[21.79, 70.87], zoom=3)
image = ee.Image("USGS/SRTMGL1_003")
vis_params = {
    "min": 0,
    "max": 6000,
    "palette": 'coolwarm', # can use 'terrain'
}
m.add_layer(image, vis_params, "SRTM")
m

Map(center=[21.79, 70.87], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

In [19]:
# For example, if we do not know the the min, max, etc of the image, we can check the data help documentation for the details from google earth.
# Anyway, lets try it to see what will happen
m = geemap.Map(center=[21.79, 70.87], zoom=3)
image = ee.Image("USGS/SRTMGL1_003")
m.add_layer(image, {}, "SRTM")
m

Map(center=[21.79, 70.87], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

**Working with ImageCollection**

For example: to create the median value image from a collection, use the `collection.median()` method. Lets create the median image from the Sentinel-2 surface reflectance collection:

We are accessing all the images from the Sentinel 2 images

In [20]:
# Loading image collection
collection = ee.ImageCollection("COPERNICUS/S2_SR")

We can simply get the median of all the images. Its becuase the max will contain mostly clouds while the min may not show the anything but grey-like

In [21]:
m = geemap.Map()
collection = ee.ImageCollection("COPERNICUS/S2_SR")

# We are processing tenns of thousands of images to create a  cloud free mosiac
image = collection.median()

vis = {
    "min": 0.0,
    "max": 3000,
    "bands": ["B4", "B3", "B2"],
}

m.set_center(83.277, 17.7009, 12) # the set center function is from GEE so it is in the lat and long format.
m.add_layer(image, vis, "Sentinel-2")
m

Map(center=[17.7009, 83.277], controls=(WidgetControl(options=['position', 'transparent_bg'], position='toprig…

**How to we filter the images?**


In [22]:
m = geemap.Map()
collection = (
    ee.ImageCollection('COPERNICUS/S2_SR')
    .filterDate('2021-01-01', '2022-01-01')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 5)) # how to find these functions? 
)
image = collection.median()

vis = {
    'min':0.0,
    'max':3000,
    'bands': ['B4', 'B3', 'B2'],

}

m.set_center(83.277, 17.7009, 12)
m.add_layer(image, vis, 'Sentinel-2')

# Check to see why it is not rendering properly
print("Images found:", collection.size().getInfo())

m


Images found: 843791


Map(center=[17.7009, 83.277], controls=(WidgetControl(options=['position', 'transparent_bg'], position='toprig…

In [23]:
# What if we know the nme of the collection but we do not know everything that is inside?
# Well, we can get the collection and only filter to see the first image:

In [24]:
collection = ee.ImageCollection('COPERNICUS/S2_SR')

In [25]:
collection.first()

## feature collection

Initially we said, an image has features and geometry. a geometry is kind of like a shape file. each row in the geometry attribute table represent a feature and columns are information for that feature in the image.

In [26]:
# So feature collection is a vector data (similar to loading a shapefile in qgis or arcgis)
m = geemap.Map()
fc = ee.FeatureCollection('TIGER/2016/Roads')
m.set_center(-73.9596, 20.7688, 12)
m.add_layer(fc, {}, 'Census roads')
m


Map(center=[20.7688, -73.9596], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topr…